In [0]:
%pip install "chromadb==0.5.23" "numpy<2.0" sentence-transformers langchain openai
dbutils.library.restartPython()

In [0]:
import os
import re
import hashlib
from pathlib import Path
from typing import List, Dict

from sentence_transformers import SentenceTransformer
import chromadb

print("All imports OK")

In [0]:
model = SentenceTransformer("BAAI/bge-small-en-v1.5")
test = model.encode(["hello world"])
print(f"Model works — vector shape: {test.shape}")

In [0]:
import os

# Use /tmp — always writable on Databricks free tier
CHROMA_PATH = "/tmp/portfolio_assistant/chroma_store"

os.makedirs(CHROMA_PATH, exist_ok=True)

chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)

collection = chroma_client.get_or_create_collection(
    name="knowledge_base",
    metadata={"hnsw:space": "cosine"}
)

print(f"Collection ready — currently has {collection.count()} chunks")

In [0]:
def chunk_markdown(text: str, source: str) -> List[Dict]:
    """
    Split markdown into sections by ## heading.
    Each section = one chunk = one vector in the store.

    Why ## and not # ?
    - # is usually the document title — too broad to be a useful chunk
    - ## sections are the right granularity: one idea each
    """
    chunks = []
    sections = re.split(r'\n(?=## )', text.strip())

    for section in sections:
        if not section.strip():
            continue

        lines = section.strip().split('\n')
        heading = lines[0].replace('#', '').strip() if lines[0].startswith('#') else "intro"
        content = '\n'.join(lines).strip()

        if len(content) < 100:
            continue

        # Stable ID — same file + heading always produces same ID
        # This means re-running ingestion updates existing chunks (upsert)
        # rather than creating duplicates
        chunk_id = hashlib.md5(f"{source}::{heading}".encode()).hexdigest()

        chunks.append({
            "id":      chunk_id,
            "text":    content,
            "source":  source,
            "section": heading,
            "type":    "project" if "projects/" in source else "cv"
        })

    return chunks

In [0]:
def embed_and_store(chunks: List[Dict]):
    """
    Embed all chunks in one batch, then upsert to Chroma.

    Chroma's upsert() = insert if ID is new, update if ID exists.
    Safe to run multiple times on the same file.
    """
    if not chunks:
        print("  No chunks to store")
        return

    texts   = [c["text"] for c in chunks]
    vectors = model.encode(texts, convert_to_numpy=True, show_progress_bar=True)

    collection.upsert(
        ids        = [c["id"]     for c in chunks],
        embeddings = [v.tolist()  for v in vectors],
        documents  = [c["text"]   for c in chunks],
        metadatas  = [{
            "source":  c["source"],
            "section": c["section"],
            "type":    c["type"]
        } for c in chunks]
    )
    print(f"  ✓ Upserted {len(chunks)} chunks")

In [0]:
def ingest_file(filepath: str):
    filepath = Path(filepath)
    print(f"\nIngesting {filepath.name}...")

    text   = filepath.read_text(encoding="utf-8")
    source = filepath.name  # e.g. "portfolio_site.md"
    chunks = chunk_markdown(text, source)

    print(f"  Found {len(chunks)} chunks")
    embed_and_store(chunks)

In [0]:
# Delete the collection and recreate it clean
chroma_client.delete_collection("knowledge_base")
collection = chroma_client.get_or_create_collection(
    name="knowledge_base",
    metadata={"hnsw:space": "cosine"}
)
print("Collection wiped")


In [0]:
# Upload portfolio_site.md to Databricks first (see note below)
# Then point this at the path

ingest_file("/Workspace/Users/ariamostajeran99@gmail.com/portfolio-assistant/knowledge/projects/portfolio_site.md")

print(f"\nTotal chunks in store: {collection.count()}")

In [0]:
def retrieve(question: str, n_results: int = 3):
    """
    Embed the question, find the most similar chunks.
    This is the core of RAG — everything else is built on top of this.
    """
    query_vector = model.encode([question]).tolist()

    results = collection.query(
        query_embeddings = query_vector,
        n_results        = n_results,
        include          = ["documents", "metadatas", "distances"]
    )

    print(f"Query: '{question}'\n")
    for i, (doc, meta, dist) in enumerate(zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    )):
        # Distance is cosine distance — lower = more similar
        # Convert to similarity score: 1 - distance
        similarity = round((1 - dist) * 100, 1)
        print(f"Result {i+1} — {similarity}% match")
        print(f"  Source: {meta['source']} / {meta['section']}")
        print(f"  Text preview: {doc[:150]}...")
        print()

# Test it
retrieve("what tech stack does Aria use for the portfolio website?")
retrieve("how does the data pipeline work?")

In [0]:
# Upload portfolio_site.md to Databricks first (see note below)
# Then point this at the path

ingest_file("/Workspace/Users/ariamostajeran99@gmail.com/portfolio-assistant/knowledge/cv.md")

print(f"\nTotal chunks in store: {collection.count()}")

In [0]:
retrieve("does Aria have experience with Docker?")
retrieve("what makes this portfolio different from a normal CV?")

In [0]:
print(f"Collection ready — currently has {collection.count()} chunks")

In [0]:
import shutil, os

def save_to_dbfs():
    """Zip the chroma store to /dbfs/ so it survives cluster restarts"""
    shutil.make_archive(
        "/Workspace/Users/ariamostajeran99@gmail.com/portfolio-assistant/02_generation",
        "zip",
        "/Workspace/Users/ariamostajeran99@gmail.com/portfolio-assistant/"
    )
    print("✓ Saved to /dbfs/portfolio_assistant/chroma_backup.zip")


In [0]:
save_to_dbfs()